In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

import matplotlib.patches as patches
fontsize = 7
plt.rcParams.update({
    'font.family': 'Arial',     # or 'Helvetica' if embedded properly
    'font.size': fontsize,
    'pdf.fonttype': 42,         # Embed TrueType (for editable text)
    'ps.fonttype': 42,
    'axes.labelsize': fontsize,
    'xtick.labelsize': fontsize,
    'ytick.labelsize': fontsize,
    'axes.facecolor': 'None',   # transparent background if needed
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.5
})


In [ ]:
dir0 = "../"
dir1 = dir0+'data/'
dirf = dir0+'figures/'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def get_payoff_matrix(r_vals, f, k, dim='2d'):
    """Computes A_ij: the fraction of strain i after competing with strain j."""
    N = len(r_vals)
    A = np.zeros((N, N))
    
    for i in range(N):
        for j in range(N):
            if i == j:
                A[i, j] = 0.5
                continue
                
            is_i_fast = r_vals[i] > r_vals[j]
            rf = r_vals[i] if is_i_fast else r_vals[j]
            rs = r_vals[j] if is_i_fast else r_vals[i]
            
            afs = k * (rs - 1)
            asf = k * (rf - 1)
            dr = rf - rs
            
            s0 = dr + afs
            s1 = dr - asf
            
            if s0 > 0 and s1 < 0:
                fast_frac = (dr + afs) / (afs + asf)
            else:
                if dim == '0d':
                    if s0 > 0 and s1 >= 0: fast_frac = 1.0
                    elif s0 <= 0 and s1 < 0: fast_frac = 0.0
                    else: 
                        phi_fast = f[i]/(f[i]+f[j]) if is_i_fast else f[j]/(f[i]+f[j])
                        f_unst = (dr + afs) / (afs + asf)
                        fast_frac = 1.0 if phi_fast > f_unst else 0.0
                        
                elif dim == '1d':
                    if afs > asf - 2 * dr: fast_frac = 1.0
                    else: fast_frac = 0.0
                        
                elif dim == '2d':
                    if s1 >= 0: fast_frac = 1.0
                    else: fast_frac = 0.0 

            A[i, j] = fast_frac if is_i_fast else (1.0 - fast_frac)
            
    return A

def simulate_competition(k=2.5, dim='2d', N=21, rw=0.5, max_steps=500, alpha=0.1):
    r_vals = np.linspace(1 - rw, 1 + rw, N)
    f = np.ones(N) / N  
    
    history = [f.copy()]
    
    for step in range(max_steps):
        A = get_payoff_matrix(r_vals, f, k, dim)
        F_target = 2 * f * (A @ f)  
        f = (1 - alpha) * f + alpha * F_target
        
        f = np.clip(f, 0, 1)
        f /= np.sum(f)
        
        history.append(f.copy())
        
    return r_vals, np.array(history)


In [ ]:

# ==========================================
# Run and Plot the 3 Dimensions
# ==========================================
k_test = 1.5
N_strains = 21
dims = ['0d', '1d', '2d']
titles = ['0D (Well-mixed)', '1D (Channel)', '2D (Expansion)']
titles = ['Growth-phenotype dominates', 'Growth-phenotype dominates', 'Growth-phenotype dominates']

fig, axes = plt.subplots(1, 3, figsize=(6, 1.8), sharey=True)

# Generate a continuous colormap from blue to red
colors = plt.cm.coolwarm(np.linspace(0, 1, N_strains))

for ax, dim, title in zip(axes, dims, titles):
    r_vals, history = simulate_competition(k=k_test, dim=dim, N=N_strains, max_steps=300, alpha=0.1)
    
    # Use the explicit colors here!
    ax.stackplot(range(len(history)), history.T, labels=[f"r={r:.2f}" for r in r_vals], colors=colors)
    
    ax.set_title(title, fontsize=7)
    ax.set_xlabel("Iteration Steps", fontsize=7)
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 1)
    
    if dim == '0d':
        ax.set_ylabel("Population Fraction", fontsize=7)

fig.savefig(dirf + f"F5_k{k_test}.pdf", bbox_inches='tight', transparent=True)

In [ ]:

# ==========================================
# Run and Plot the 3 Dimensions
# ==========================================
k_test = 2.5
N_strains = 21
dims = ['0d', '1d', '2d']
titles = ['0D (Well-mixed)', '1D (Channel)', '2D (Expansion)']
titles = ['Inhibition-phenotype dominates', 'Inhibition-phenotype dominates', 'Balanced']

fig, axes = plt.subplots(1, 3, figsize=(6, 1.8), sharey=True)

# Generate a continuous colormap from blue to red
colors = plt.cm.coolwarm(np.linspace(0, 1, N_strains))

for ax, dim, title in zip(axes, dims, titles):
    r_vals, history = simulate_competition(k=k_test, dim=dim, N=N_strains, max_steps=300, alpha=0.1)
    
    # Use the explicit colors here!
    ax.stackplot(range(len(history)), history.T, labels=[f"r={r:.2f}" for r in r_vals], colors=colors)
    
    ax.set_title(title, fontsize=7)
    ax.set_xlabel("Iteration Steps", fontsize=7)
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 1)
    
    if dim == '0d':
        ax.set_ylabel("Population Fraction", fontsize=7)

fig.savefig(dirf + f"F5_k{k_test}.pdf", bbox_inches='tight', transparent=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(0.1, 2.0))
gradient = np.linspace(0, 1, 257)
gradient = np.vstack((gradient, gradient)).T

# Extract exactly 4 discrete colors from the continuous 'coolwarm' map
discrete_cmap = plt.get_cmap('coolwarm', 8)

# Apply the discrete colormap
ax.imshow(gradient, aspect='auto', cmap=discrete_cmap, origin='lower')

ax.set_xticks([])
ax.set_yticks([0, 256])
ax.set_yticklabels(['Slow', 'Fast'])

# fig.savefig(dirf + 'cbar.pdf', bbox_inches='tight', transparent=True)
plt.show()